In [ ]:
import os
from PIL import Image
import numpy as np
import pandas as pd
import tensorflow as tf

MAIN_DIR = "/home/onuro/code/simonwilliams32/MRI_project"

FINAL_DATASET_DIR = os.path.join(MAIN_DIR, "raw_data", "final_brain_tumor_preprocessed_dataset")

TRAIN_DIR = FINAL_DATASET_DIR + "/train"
VAL_DIR = FINAL_DATASET_DIR + "/val"
TEST_DIR = FINAL_DATASET_DIR + "/test"
EXTERNAL_VAL_DIR = FINAL_DATASET_DIR + "/external_val"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

class_names = ["glioma", "meningioma", "notumor", "pituitary"]
IMAGE_EXTS = (".png", ".jpg", ".jpeg")

##### Check for / convert colorized images into grayscale

def is_grayscale(path):
    """Returns True if image is black & white (all RGB channels equal)."""
    try:
        img = Image.open(path).convert("RGB")
        arr = np.array(img)
        return np.array_equal(arr[:, :, 0], arr[:, :, 1]) and np.array_equal(arr[:, :, 1], arr[:, :, 2])
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None

results = []

for split in os.listdir(FINAL_DATASET_DIR):
    split_path = os.path.join(FINAL_DATASET_DIR, split)
    if not os.path.isdir(split_path):
        continue

    for class_name in os.listdir(split_path):
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue

        for fname in os.listdir(class_path):
            if not fname.lower().endswith(IMAGE_EXTS):
                continue
            fpath = os.path.join(class_path, fname)
            gray = is_grayscale(fpath)
            results.append({
                "split": split,
                "class": class_name,
                "filename": fname,
                "path": fpath,
                "is_grayscale": gray
            })

df = pd.DataFrame(results)

print(f"Total images checked: {len(df)}")
print(f"Grayscale: {(df['is_grayscale'] == True).sum()}")
print(f"Colormapped/Color: {(df['is_grayscale'] == False).sum()}")
print(f"Errors/unreadable: {df['is_grayscale'].isna().sum()}")

print("\nBreakdown by split and class:")
summary = df.groupby(["split", "class"])["is_grayscale"].value_counts().unstack(fill_value=0)
print(summary)


def convert_to_grayscale_and_save(path):
    """Convert an image to grayscale (L), then save it back as 3-channel RGB
    so shape/format stays consistent with the rest of the pipeline."""
    try:
        img = Image.open(path).convert("RGB")
        gray = img.convert("L")
        gray_rgb = Image.merge("RGB", (gray, gray, gray))
        gray_rgb.save(path)
        return True
    except Exception as e:
        print(f"Error converting {path}: {e}")
        return False

# Only touch the ones flagged as colorized
to_fix = df[df["is_grayscale"] == False]
print(f"Converting {len(to_fix)} colorized images to grayscale...")

converted = 0
failed = 0
for path in to_fix["path"]:
    if convert_to_grayscale_and_save(path):
        converted += 1
    else:
        failed += 1

print(f"Converted: {converted}")
print(f"Failed: {failed}")


# load datasets

train_ds = tf.keras.utils.image_dataset_from_directory(
     TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
     label_mode="categorical", shuffle=True, class_names=class_names
 )
val_ds = tf.keras.utils.image_dataset_from_directory(
     VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
     label_mode="categorical", shuffle=False, class_names=class_names
 )
test_ds = tf.keras.utils.image_dataset_from_directory(
     TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
     label_mode="categorical", shuffle=False, class_names=class_names
)

# external validation is not used for this model
""" external_val_ds = tf.keras.utils.image_dataset_from_directory(
     EXTERNAL_VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
     label_mode="categorical", shuffle=False, class_names=class_names
 )
 """

print("Training paths prepared:")
print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)
print("TEST_DIR:", TEST_DIR)
print("EXTERNAL_VAL_DIR:", EXTERNAL_VAL_DIR)

I0000 00:00:1784374938.044254    1977 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784374938.044843    1977 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784374938.376501    1977 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784374941.725688    1977 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

Total images checked: 14602
Grayscale: 14602
Colormapped/Color: 0
Errors/unreadable: 0

Breakdown by split and class:
is_grayscale      True
split class           
test  glioma       862
      meningioma   665
      notumor      260
      pituitary    404
train glioma      4018
      meningioma  3104
      notumor     1212
      pituitary   1886
val   glioma       862
      meningioma   665
      notumor      260
      pituitary    404
Converting 0 colorized images to grayscale...
Converted: 0
Failed: 0
Found 10220 files belonging to 4 classes.
Found 2191 files belonging to 4 classes.
Found 2191 files belonging to 4 classes.


E0000 00:00:1784374974.474901    1977 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Training paths prepared:
TRAIN_DIR: /home/onuro/code/simonwilliams32/MRI_project/raw_data/final_brain_tumor_preprocessed_dataset/train
VAL_DIR: /home/onuro/code/simonwilliams32/MRI_project/raw_data/final_brain_tumor_preprocessed_dataset/val
TEST_DIR: /home/onuro/code/simonwilliams32/MRI_project/raw_data/final_brain_tumor_preprocessed_dataset/test
EXTERNAL_VAL_DIR: /home/onuro/code/simonwilliams32/MRI_project/raw_data/final_brain_tumor_preprocessed_dataset/external_val


In [2]:
# data needs to be scaled in the way the transferred model requires
# skip this part for transfer model
""" #SCALING!
#Inspect Shape
images, labels = next(iter(train_ds))
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("\nFirst image shape:", images[0].shape)
print("First label:", labels[0])

#Next block
#check pixel values
for images, labels in train_ds.take(5):
    print(images.numpy().min(),
          images.numpy().max())

#Next block
#Scale down pixel values
train_ds_scaled = train_ds.map(lambda images, labels: (images / 255.0, labels))
val_ds_scaled = val_ds.map(lambda images, labels: (images / 255.0, labels))
test_ds_scaled = test_ds.map(lambda images, labels: (images / 255.0, labels))
external_val_ds_scaled = external_val_ds.map(lambda images, labels: (images / 255.0, labels))

#Next block
#Check that scaling has been done (shouls be between 0 and 1)
images, labels = next(iter(train_ds_scaled))
print(images.numpy().min())
print(images.numpy().max())
print(images.dtype) """

' #SCALING!\n#Inspect Shape\nimages, labels = next(iter(train_ds))\nprint("Images shape:", images.shape)\nprint("Labels shape:", labels.shape)\nprint("Image dtype:", images.dtype)\nprint("\nFirst image shape:", images[0].shape)\nprint("First label:", labels[0])\n\n#Next block\n#check pixel values\nfor images, labels in train_ds.take(5):\n    print(images.numpy().min(),\n          images.numpy().max())\n\n#Next block\n#Scale down pixel values\ntrain_ds_scaled = train_ds.map(lambda images, labels: (images / 255.0, labels))\nval_ds_scaled = val_ds.map(lambda images, labels: (images / 255.0, labels))\ntest_ds_scaled = test_ds.map(lambda images, labels: (images / 255.0, labels))\nexternal_val_ds_scaled = external_val_ds.map(lambda images, labels: (images / 255.0, labels))\n\n#Next block\n#Check that scaling has been done (shouls be between 0 and 1)\nimages, labels = next(iter(train_ds_scaled))\nprint(images.numpy().min())\nprint(images.numpy().max())\nprint(images.dtype) '

In [3]:
# skip this part as well
# transfer model uses 3 channels, so keep all

""" #convert every image to one-channel grayscale (instead of RGB because we already have gray images)
# this decreases the data size 3X, makes the analysis faster

train_ds_scaled = train_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))
val_ds_scaled = val_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))
test_ds_scaled = test_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))
external_val_ds_scaled = external_val_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y)) """

' #convert every image to one-channel grayscale (instead of RGB because we already have gray images)\n# this decreases the data size 3X, makes the analysis faster\n\ntrain_ds_scaled = train_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))\nval_ds_scaled = val_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))\ntest_ds_scaled = test_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))\nexternal_val_ds_scaled = external_val_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y)) '

In [ ]:
from tensorflow.keras.applications import efficientnet

AUTOTUNE = tf.data.AUTOTUNE
# EfficientNet requires a specific scaling, expects raw 0-255 input

def prepare_for_efficientnet(images, labels):
    images = efficientnet.preprocess_input(images)
    return images, labels

###### Train ######
train_ds_effnet_multi = train_ds.map(prepare_for_efficientnet)
train_ds_effnet_multi = train_ds_effnet_multi.cache()
train_ds_effnet_multi = train_ds_effnet_multi.shuffle(1000)
train_ds_effnet_multi = train_ds_effnet_multi.prefetch(AUTOTUNE)

###### Validation ######
val_ds_effnet_multi = val_ds.map(prepare_for_efficientnet)
val_ds_effnet_multi = val_ds_effnet_multi.prefetch(AUTOTUNE)

###### Test ######
test_ds_effnet_multi = test_ds.map(prepare_for_efficientnet)
test_ds_effnet_multi = test_ds_effnet_multi.prefetch(AUTOTUNE)

""" ###### External validation ######
external_val_ds_effnet_multi = external_val_ds.map(prepare_for_efficientnet)
external_val_ds_effnet_multi = external_val_ds_effnet_multi.prefetch(AUTOTUNE)
 """


images, labels = next(iter(train_ds_effnet_multi))
print("Image batch shape:", images.shape) # (32, 224, 224, 3)
print("Label batch shape:", labels.shape) # (32, 4)
print(labels[:5])


W0000 00:00:1784374975.184943    2342 png_io.cc:96] PNG warning: iCCP: known incorrect sRGB profile
W0000 00:00:1784374975.484300    2339 png_io.cc:96] PNG warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


Image batch shape: (32, 224, 224, 3)
Label batch shape: (32, 4)
tf.Tensor(
[[1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]], shape=(5, 4), dtype=float32)


In [ ]:
import tensorflow as tf
tf.keras.backend.clear_session()

base_model = efficientnet.EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


In [ ]:
from keras import layers, models


def add_last_layers_multi(base_model):
    '''Take a pre-trained model, set its parameters as non-trainable, and add additional trainable layers'''
    base_model.trainable = False

    pooling_layer = layers.GlobalAveragePooling2D()
    dense_layer = layers.Dense(64, activation="relu")
    dropout_layer = layers.Dropout(0.4)
    prediction_layer = layers.Dense(4, activation="softmax")  # multiclass prediction, use softmax

    model = tf.keras.Sequential([
        base_model,
        pooling_layer,
        dense_layer,
        dropout_layer,
        prediction_layer
    ])
    return model


In [ ]:
from tensorflow.keras import optimizers

def build_model_multi(base_model):
    model = add_last_layers_multi(base_model)
    model.compile(
        loss="categorical_crossentropy", # this is for multi-class prediction
        optimizer=optimizers.Adam(learning_rate=1e-4),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.Precision(name="precision"),
        ]
    )
    return model

model = build_model_multi(base_model)


In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,131,815 (15.76 MB)

 Trainable params: 82,244 (321.27 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [9]:
from keras.callbacks import EarlyStopping

es = EarlyStopping(
    monitor="val_recall",
    mode="max",
    patience=8,
    restore_best_weights=True
)


In [10]:
from sklearn.utils.class_weight import compute_class_weight

y_train = []
for images, labels in train_ds:
    class_idx = tf.argmax(labels, axis=1).numpy()
    y_train.extend(class_idx)
y_train = np.array(y_train)

print("Class counts in train:")
for i, name in enumerate(class_names):
    print(f"{name}: {(y_train == i).sum()}")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)


W0000 00:00:1784374980.775431    2377 png_io.cc:96] PNG warning: iCCP: known incorrect sRGB profile
W0000 00:00:1784374980.966972    2374 png_io.cc:96] PNG warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


Class counts in train:
glioma: 4018
meningioma: 3104
notumor: 1212
pituitary: 1886
{0: np.float64(0.6358885017421603), 1: np.float64(0.823131443298969), 2: np.float64(2.1080858085808583), 3: np.float64(1.3547189819724283)}


In [11]:
history = model.fit(
    train_ds_effnet_multi,
    validation_data=val_ds_effnet_multi,
    epochs=100,
    callbacks=[es],
    class_weight=class_weight_dict,
)


Epoch 1/100
320/320 ━━━━━━━━━━━━━━━━━━━━ 207s 629ms/step - accuracy: 0.6477 - loss: 0.7643 - precision: 0.7832 - recall: 0.4614 - val_accuracy: 0.8211 - val_loss: 0.5218 - val_precision: 0.8686 - val_recall: 0.7540
Epoch 2/100
320/320 ━━━━━━━━━━━━━━━━━━━━ 204s 639ms/step - accuracy: 0.7994 - loss: 0.4631 - precision: 0.8409 - recall: 0.7448 - val_accuracy: 0.8549 - val_loss: 0.4153 - val_precision: 0.8821 - val_recall: 0.8165
Epoch 3/100
320/320 ━━━━━━━━━━━━━━━━━━━━ 263s 824ms/step - accuracy: 0.8292 - loss: 0.3885 - precision: 0.8552 - recall: 0.7913 - val_accuracy: 0.8686 - val_loss: 0.3601 - val_precision: 0.8865 - val_recall: 0.8448
Epoch 4/100
320/320 ━━━━━━━━━━━━━━━━━━━━ 204s 636ms/step - accuracy: 0.8495 - loss: 0.3529 - precision: 0.8721 - recall: 0.8223 - val_accuracy: 0.8745 - val_loss: 0.3391 - val_precision: 0.8928 - val_recall: 0.8590
Epoch 5/100
320/320 ━━━━━━━━━━━━━━━━━━━━ 192s 600ms/step - accuracy: 0.8607 - loss: 0.3160 - precision: 0.8782 - recall: 0.8403 - val_accura

In [12]:
model.evaluate(test_ds_effnet_multi)

69/69 ━━━━━━━━━━━━━━━━━━━━ 34s 486ms/step - accuracy: 0.9658 - loss: 0.0898 - precision: 0.9662 - recall: 0.9653


[0.08981318026781082,
 0.9657690525054932,
 0.965312659740448,
 0.9661946296691895]

In [13]:
#model.evaluate(external_val_ds_effnet_multi)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_model_multi(model, dataset, class_names=class_names):

    y_true = []
    y_pred = []
    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(tf.argmax(labels, axis=1).numpy())
        y_pred.extend(tf.argmax(preds, axis=1).numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    print("classification Report")
    print(classification_report(
        y_true, y_pred,
        labels=range(len(class_names)),
        target_names=class_names,
        zero_division=0
    ))

    print("Confusion Matrix")
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix")
    plt.show()

    return y_true, y_pred


y_true_test, y_pred_test = evaluate_model_multi(model, test_ds_effnet_multi)


In [ ]:
# this part is not needed as we don't use external validation
# External validation
# only has glioma/meningioma
#y_true_ext, y_pred_ext = evaluate_model_multi(model, external_val_ds_effnet_multi, dataset_name="External Validation")


In [ ]:
# save the model

#model.save("brain_tumor_model_4class_transferlearning.keras")